# 01. Extract CLAP audio + text embeddings

We use the **LAION-CLAP** model (HTSAT-tiny audio encoder + RoBERTa text encoder, the `enable_fusion=False` variant — same checkpoint shipped by `laion_clap.CLAP_Module().load_ckpt()`).

Inputs are taken from `data/esc50_metadata.csv` produced by `src/download_esc50.py`. Outputs are saved to `results/clap/`.

We extract:
- audio embeddings: shape (N, 512), L2-normalizable for cosine similarity
- text embeddings:  shape (N, 512), one per caption (`'This is a sound of {class_name}.'`)
- a per-class text embedding table (50 prompts) for zero-shot classification later


In [ ]:
import os, sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

DATA_CSV    = PROJECT_DIR / 'data' / 'esc50_metadata.csv'
RESULTS_DIR = PROJECT_DIR / 'results' / 'clap'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('CSV exists  :', DATA_CSV.exists())
print('CSV path    :', DATA_CSV)

## Load metadata and filter unreadable files

In [ ]:
import pandas as pd
import numpy as np
import soundfile as sf

df = pd.read_csv(DATA_CSV)
print('rows in CSV:', len(df))

good = []
for p in df['audio_path']:
    try:
        data, _ = sf.read(p)
        good.append(len(data) > 0)
    except Exception:
        good.append(False)
df = df[good].reset_index(drop=True)
print('rows after filtering:', len(df))
df.head()

## Load LAION-CLAP

First call auto-downloads the default checkpoint into `~/.cache/`.

In [ ]:
import laion_clap

model = laion_clap.CLAP_Module(enable_fusion=False)
model.load_ckpt()
print('CLAP model loaded.')

## Extract audio embeddings

`get_audio_embedding_from_filelist` handles its own batching internally.

In [ ]:
audio_paths = df['audio_path'].tolist()
captions    = df['caption'].tolist()

audio_emb = model.get_audio_embedding_from_filelist(x=audio_paths, use_tensor=False)
print('audio_emb shape:', audio_emb.shape, audio_emb.dtype)

## Extract per-caption text embeddings

These are aligned 1-to-1 with the audio embeddings (used for paired similarity + retrieval).

In [ ]:
text_emb = model.get_text_embedding(captions, use_tensor=False)
print('text_emb shape:', text_emb.shape, text_emb.dtype)

## Extract per-class text embeddings (50 prompts) for zero-shot

Same prompt template as the row-level captions, so audio  →  argmax over 50 prompts is a fair zero-shot setup.

In [ ]:
class_names    = sorted(df['class_name'].unique().tolist())
class_prompts  = [f'This is a sound of {n.replace("_", " ")}.' for n in class_names]
class_text_emb = model.get_text_embedding(class_prompts, use_tensor=False)
print('class_names    :', class_names[:5], '...')
print('class_text_emb :', class_text_emb.shape)

## Save outputs

In [ ]:
np.save(RESULTS_DIR / 'audio_embeddings.npy', audio_emb)
np.save(RESULTS_DIR / 'text_embeddings.npy',  text_emb)
np.save(RESULTS_DIR / 'class_text_embeddings.npy', class_text_emb)
with open(RESULTS_DIR / 'class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))
df.to_csv(RESULTS_DIR / 'metadata.csv', index=False)

print('saved to', RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print(' ', p.name, p.stat().st_size, 'bytes')